CareerPilot AI – Career & Internship Guidance Assistant


A conversational AI agent that helps college students with career guidance, skill recommendations, internship guidance, resume improvement, interview preparation, and internship-related complaints. Run in Google Colab and add your Groq API key in Colab Secrets as GROQ_API_KEY.

1. **Install libraries**

In [3]:
!pip install -q openai ipywidgets

2. Connect to Groq API

In [15]:
import getpass
from openai import OpenAI
from google.colab import userdata

try:
    api_key = userdata.get("GROQ_API_KEY")
    if not api_key or api_key in ["gsk_0", "gst_0"]:
        raise ValueError("Placeholder key detected")
except Exception:
    print("Groq API Key secret not found in Colab Secrets or placeholder detected.")
    api_key = getpass.getpass("Please enter your actual Groq API Key (starts with gsk_): ")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)

# Note: Common Groq models are "llama3-8b-8192", "llama3-70b-8192", or "mixtral-8x7b-32768"
MODEL_NAME = "llama3-8b-8192"

print("Client configuration updated! Please run this cell and enter your valid API key to proceed.")

Client configuration updated! Please run this cell and enter your valid API key to proceed.


In [20]:
from openai import OpenAI
from google.colab import userdata

api_key = userdata.get("GROQ_API_KEY")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)

MODEL_NAME = "openai/gpt-oss-120b"

print("CareerPilot AI Connected!")

CareerPilot AI Connected!


In [21]:
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": "Hello"}
    ]
)

print(response.choices[0].message.content)

Hello! How can I assist you today?


3. Define sample career information

In [6]:
CAREER_INFO = {
    "python_developer": {
        "skills": "Python, SQL, OOP, DSA",
        "roles": "Python Developer, Backend Developer"
    },

    "data_scientist": {
        "skills": "Python, Statistics, Machine Learning, SQL",
        "roles": "Data Scientist, ML Engineer"
    },

    "cloud_engineer": {
        "skills": "AWS, Linux, Networking, Docker",
        "roles": "Cloud Engineer, DevOps Engineer"
    },

    "web_developer": {
        "skills": "HTML, CSS, JavaScript, React",
        "roles": "Frontend Developer, Full Stack Developer"
    },

    "internship": "Students can search for internships through company career pages and verified internship platforms.",

    "resume": "A resume should include education, technical skills, projects, certifications and achievements.",

    "interview": "Prepare programming fundamentals, technical concepts, projects and communication skills.",

    "career_guidance": "Choose a career based on your interests, strengths and skills."
}

print(CAREER_INFO)

{'python_developer': {'skills': 'Python, SQL, OOP, DSA', 'roles': 'Python Developer, Backend Developer'}, 'data_scientist': {'skills': 'Python, Statistics, Machine Learning, SQL', 'roles': 'Data Scientist, ML Engineer'}, 'cloud_engineer': {'skills': 'AWS, Linux, Networking, Docker', 'roles': 'Cloud Engineer, DevOps Engineer'}, 'web_developer': {'skills': 'HTML, CSS, JavaScript, React', 'roles': 'Frontend Developer, Full Stack Developer'}, 'internship': 'Students can search for internships through company career pages and verified internship platforms.', 'resume': 'A resume should include education, technical skills, projects, certifications and achievements.', 'interview': 'Prepare programming fundamentals, technical concepts, projects and communication skills.', 'career_guidance': 'Choose a career based on your interests, strengths and skills.'}


4. Create the system prompt

In [7]:
import json

SYSTEM_PROMPT = f"""
ROLE:
You are CareerPilot AI, a friendly career and internship guidance assistant.

AVAILABLE INFORMATION:
{json.dumps(CAREER_INFO)}

RULES:
1. Help students understand different career paths.
2. Suggest skills based on the student's career interests.
3. Provide internship search guidance.
4. Help students improve their resumes.
5. Provide basic interview preparation guidance.
6. Never invent company internship openings or job vacancies.
7. Never guarantee internship or job selection.
8. If information is unavailable, clearly mention it.
9. Keep answers simple and student-friendly.
10. Respond in Telugu if the student requests it.
11. Remember the context of the conversation.
"""

print("System prompt created!")

System prompt created!


5. Classify the student's intent

In [24]:
# Use the verified working model from your previous connection cell
MODEL_NAME = "openai/gpt-oss-120b"


def classify_intent(message: str) -> str:
    messages = [
        {
            "role": "system",
            "content": """
            Classify the message into exactly one category:
            career_guidance, skill_guidance, internship,
            resume, interview, complaint.
            Return only the category.
            """
        },
        {"role": "user", "content": "Which career is suitable for me?"},
        {"role": "assistant", "content": "career_guidance"},
        {
            "role": "user",
            "content": "What skills are required for cloud computing?",
        },
        {"role": "assistant", "content": "skill_guidance"},
        {"role": "user", "content": "How can I find internships?"},
        {"role": "assistant", "content": "internship"},
        {"role": "user", "content": "How can I improve my resume?"},
        {"role": "assistant", "content": "resume"},
        {"role": "user", "content": "Give me interview questions?"},
        {"role": "assistant", "content": "interview"},
        {
            "role": "user",
            "content": "The internship application website is not working.",
        },
        {"role": "assistant", "content": "complaint"},
        {"role": "user", "content": message},
    ]

    response = client.chat.completions.create(
        model=MODEL_NAME, temperature=0, messages=messages
    )

    return response.choices[0].message.content.strip().lower()


print(classify_intent("What skills should I learn for data science?"))

skill_guidance


6. Extract internship complaint details as JSON

In [26]:
def extract_complaint(message: str) -> dict:

    prompt = """
    Extract internship-related complaint details.

    Return only valid JSON with these keys:

    {
        "student_name": "string or null",
        "company_name": "string or null",
        "issue": "string",
        "application_status": "string or null",
        "priority": "low, medium or high"
    }

    Do not invent missing information.
    Use null when unknown.
    """

    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0,
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": message}
        ]
    )

    try:
        return json.loads(
            response.choices[0].message.content
        )

    except json.JSONDecodeError:
        return {"error": "Invalid JSON response"}


sample = """
Hi, I am Ananya.
I applied for an internship at ABC Technologies.
My application status is not updating for five days.
"""

result = extract_complaint(sample)

print(json.dumps(result, indent=2))

{
  "student_name": "Ananya",
  "company_name": "ABC Technologies",
  "issue": "Application status not updating for five days",
  "application_status": null,
  "priority": "medium"
}


7. Create a logging decorator

In [27]:
import functools
import time

def log_call(func):

    @functools.wraps(func)
    def wrapper(*args, **kwargs):

        start = time.time()

        print("Running:", func.__name__)

        result = func(*args, **kwargs)

        print(
            "Execution time:",
            round(time.time() - start, 4),
            "seconds"
        )

        return result

    return wrapper


@log_call
def recommend_skills(career: str) -> dict:

    career = career.lower().replace(" ", "_")

    if career not in CAREER_INFO:
        return {"error": "Career information not available."}

    return CAREER_INFO[career]


print(recommend_skills("cloud_engineer"))

Running: recommend_skills
Execution time: 0.0001 seconds
{'skills': 'AWS, Linux, Networking, Docker', 'roles': 'Cloud Engineer, DevOps Engineer'}


8. Build the agent with conversation memory

In [28]:
class CareerPilotAgent:

    def __init__(self, system_prompt, max_turns=5):

        self.system_prompt = system_prompt
        self.max_turns = max_turns

        self.reset()

    def reset(self):

        self.history = [
            {"role": "system", "content": self.system_prompt}
        ]

    def chat(self, message: str) -> str:

        if not message.strip():
            return "Please enter a message."

        self.history.append(
            {"role": "user", "content": message}
        )

        messages = [
            self.history[0]
        ] + self.history[1:][-(self.max_turns * 2):]

        try:

            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=messages,
                temperature=0.3
            )

            reply = response.choices[0].message.content or ""

        except Exception as e:

            print("API error:", e)

            reply = "Sorry, CareerPilot AI is temporarily unavailable."

        self.history.append(
            {"role": "assistant", "content": reply}
        )

        return reply


agent = CareerPilotAgent(SYSTEM_PROMPT)

print("CareerPilot AI Agent Ready!")

CareerPilot AI Agent Ready!


9. Test the agent and its memory

In [30]:
print(agent.chat("Hi, my name is Prasanna."))

print(agent.chat("I am interested in cloud computing."))

print(agent.chat("What skills should I learn?"))

print(agent.chat("Which career roles can I target?"))

Hi Prasanna! 👋  
Nice to meet you. How can I help you today?

- Exploring cloud‑computing career paths?  
- Building or polishing your resume?  
- Finding internships?  
- Getting ready for interviews?  

Just let me know what you’d like to focus on!
Nice to meet you, Prasanna! 🎉  
Here’s a quick, student‑friendly guide to get you started in **cloud computing**.

---

## 1️⃣ Cloud‑Computing Career Paths
| Role | What You’ll Do | Core Skills to Master |
|------|----------------|-----------------------|
| **Cloud Engineer** | Build & manage cloud infrastructure (servers, storage, networking). | AWS/Azure/GCP services, Linux, Terraform/CloudFormation, Python/Bash, networking, IAM. |
| **DevOps Engineer** | Automate code‑to‑deployment pipelines, monitor systems, ensure reliability. | CI/CD (GitHub Actions, Jenkins), Docker, Kubernetes, IaC, scripting, monitoring (CloudWatch, Prometheus). |
| **Site Reliability Engineer (SRE)** | Keep large‑scale services up, fast, and fault‑tolerant. | Clo

10. Create an interactive chat interface

In [31]:
import ipywidgets as widgets

from IPython.display import display, clear_output


chat_output = widgets.Output()

message_input = widgets.Text(
    placeholder="Ask CareerPilot AI about careers or internships...",
    layout=widgets.Layout(width="75%")
)

send_button = widgets.Button(
    description="Send",
    button_style="primary"
)

reset_button = widgets.Button(
    description="Reset Chat",
    button_style="warning"
)


def send_message(button):

    message = message_input.value.strip()

    if not message:
        return

    message_input.value = ""

    with chat_output:

        print("You:", message)

        print("CareerPilot AI:", agent.chat(message))

        print("-" * 55)


def reset_chat(button):

    agent.reset()

    with chat_output:
        clear_output()


send_button.on_click(send_message)

reset_button.on_click(reset_chat)

message_input.on_submit(send_message)


display(
    widgets.HTML(
        "<h2>🚀 CareerPilot AI – Career Guidance Assistant</h2>"
    )
)

display(chat_output)

display(
    widgets.HBox([
        message_input,
        send_button,
        reset_button
    ])
)

HTML(value='<h2>🚀 CareerPilot AI – Career Guidance Assistant</h2>')

Output()

11. Try these sample questions

1.How can I find internships?



2.How do I improve my resume?



3.Give me five basic Python interview questions.



4.I am interested in AWS. What career roles can I target?



5.My internship application status is not updating. Can I report it?

Concepts demonstrated
- Groq API: Connects to the language model.
- System prompt: Defines the agent's role and boundaries.
- Intent classification: Identifies the student's request.
- JSON extraction: Converts complaints into structured data.
- Decorator: Logs function execution time.
- Agent class: Maintains conversation history.
- Conversation memory: Remembers previous messages.
- Error handling: Handles API errors.
- ipywidgets: Provides an interactive chat interface.